# Test your connection to the course LLM server

This notebook checks in two small steps whether your `.env` file is set up correctly and whether you can reach the course LLM server.

**Before you start:** put this notebook into your `ai-engineering-course` folder (or the `chapter` folder inside it), so that it can find your `.env` file. Then run the cells one after another.

Every step prints ✅ if everything is fine, or ❌ together with a hint on what to fix.

## Step 1: Find and read the `.env` file

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

env_path = find_dotenv(usecwd=True)
if not env_path:
    print("❌ No .env file found.")
    print("   Create it in your ai-engineering-course folder (see the credentials handout)")
    print("   and make sure this notebook is inside that folder.")
else:
    load_dotenv(env_path, override=True)
    print(f"✅ Found .env at: {env_path}")

missing = [name for name in ["LLM_HOST", "LLM_PORT", "LLM_API_KEY"] if not os.environ.get(name)]
if missing:
    print(f"❌ Missing or empty in .env: {', '.join(missing)}")
    print("   Check spelling, one variable per line, no spaces around '='.")
else:
    key = os.environ["LLM_API_KEY"]
    print(f"✅ LLM_HOST    = {os.environ['LLM_HOST']}")
    print(f"✅ LLM_PORT    = {os.environ['LLM_PORT']}")
    print(f"✅ LLM_API_KEY = {key[:2]}{'*' * max(len(key) - 2, 0)}  (hidden)")

## Step 2: Send a test prompt to both course models

We send a short prompt to the two models used in the course. The first request can take a few seconds while a model is loaded.

In [ ]:
from openai import OpenAI

LLM_URL = f"http://{os.environ['LLM_HOST']}:{os.environ['LLM_PORT']}/v1"
client = OpenAI(base_url=LLM_URL, api_key=os.environ["LLM_API_KEY"], timeout=120)

all_ok = True
for model in ["gemma3:4b", "gemma4:26b"]:
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": "Say hello to the AI Engineering course in one short sentence."}],
            temperature=0.0,
        )
        print(f"✅ {model}: {response.choices[0].message.content.strip()}")
    except Exception as e:
        all_ok = False
        print(f"❌ {model}: {type(e).__name__}: {e}")

print()
if all_ok:
    print("🎉 Everything works - you're ready for chapter 03!")
else:
    print("Hints: check LLM_HOST / LLM_PORT / LLM_API_KEY in .env (no typos, no quotes),")
    print("check your internet connection, then restart the kernel and run all cells again.")

> **Changed your `.env`?** Restart the kernel (*Kernel → Restart Kernel*) and run all cells again, so the new values are loaded.